# Best-Methods Figure


In [1]:
import warnings
from collections import defaultdict
from itertools import product
from pathlib import Path
from typing import Any, Literal, cast

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from tqdm.auto import tqdm

import qaoa_parameter_setting.utils as utils
import qaoa_parameter_setting.utils.database as database
from qaoa_parameter_setting.utils.types import (
    Depth,
    EvaluationType,
    GraphKey,
    MethodConfigJSON,
)

plt.rcParams["text.usetex"] = True
mpl.use("pdf")

In [2]:
# Here we hard-code which instances we want, so we are consistent between all
# bars and axes in the figure.

# idx chosen to reduce the number of new experiments to run, based on already
# existing runs.
idx = "001"
SMALL_INSTANCES = {
    # 10x 20-node 20-percent Erdös-Rényi graphs
    "000_20nodes_erdosrenyi20percent.json",
    "001_20nodes_erdosrenyi20percent.json",
    "002_20nodes_erdosrenyi20percent.json",
    "003_20nodes_erdosrenyi20percent.json",
    "004_20nodes_erdosrenyi20percent.json",
    "005_20nodes_erdosrenyi20percent.json",
    "006_20nodes_erdosrenyi20percent.json",
    "007_20nodes_erdosrenyi20percent.json",
    "008_20nodes_erdosrenyi20percent.json",
    "009_20nodes_erdosrenyi20percent.json",
    # 10x 1-by-2 21-node Heavy-Hex graphs
    "000_1_2_heavyhex_21nodes_weighted.json",
    "001_1_2_heavyhex_21nodes_weighted.json",
    "002_1_2_heavyhex_21nodes_weighted.json",
    "003_1_2_heavyhex_21nodes_weighted.json",
    "004_1_2_heavyhex_21nodes_weighted.json",
    "005_1_2_heavyhex_21nodes_weighted.json",
    "006_1_2_heavyhex_21nodes_weighted.json",
    "007_1_2_heavyhex_21nodes_weighted.json",
    "008_1_2_heavyhex_21nodes_weighted.json",
    "009_1_2_heavyhex_21nodes_weighted.json",
    # 1x 20-node line-to-full graph per 1 to 10 swap layers, for total of 10x graphs
    f"{idx}_20nodes_1swap_layers.json",
    f"{idx}_20nodes_2swap_layers.json",
    f"{idx}_20nodes_3swap_layers.json",
    f"{idx}_20nodes_4swap_layers.json",
    f"{idx}_20nodes_5swap_layers.json",
    f"{idx}_20nodes_6swap_layers.json",
    f"{idx}_20nodes_7swap_layers.json",
    f"{idx}_20nodes_8swap_layers.json",
    f"{idx}_20nodes_9swap_layers.json",
    f"{idx}_20nodes_10swap_layers.json",
    # 1x 20-node random-regular graph per degree d=3,4,5,6,7,8,9 for a total of 7x graphs
    f"{idx}_20nodes_random3regular.json",
    f"{idx}_20nodes_random4regular.json",
    f"{idx}_20nodes_random5regular.json",
    f"{idx}_20nodes_random6regular.json",
    f"{idx}_20nodes_random7regular.json",
    f"{idx}_20nodes_random8regular.json",
    f"{idx}_20nodes_random9regular.json",
}
LARGE_INSTANCES = {
    # 10x 50-node 20-percent Erdös-Rényi graphs
    "000_50nodes_erdosrenyi20percent.json",
    "001_50nodes_erdosrenyi20percent.json",
    "002_50nodes_erdosrenyi20percent.json",
    "003_50nodes_erdosrenyi20percent.json",
    "004_50nodes_erdosrenyi20percent.json",
    "005_50nodes_erdosrenyi20percent.json",
    "006_50nodes_erdosrenyi20percent.json",
    "007_50nodes_erdosrenyi20percent.json",
    "008_50nodes_erdosrenyi20percent.json",
    "009_50nodes_erdosrenyi20percent.json",
    # 10x 144-node Heavy-Hex graphs
    "000_7_3_heavyhex_144nodes_weighted.json",
    "001_7_3_heavyhex_144nodes_weighted.json",
    "002_7_3_heavyhex_144nodes_weighted.json",
    "003_7_3_heavyhex_144nodes_weighted.json",
    "004_7_3_heavyhex_144nodes_weighted.json",
    "005_7_3_heavyhex_144nodes_weighted.json",
    "006_7_3_heavyhex_144nodes_weighted.json",
    "007_7_3_heavyhex_144nodes_weighted.json",
    "008_7_3_heavyhex_144nodes_weighted.json",
    "009_7_3_heavyhex_144nodes_weighted.json",
    # 1x 100-node line-to-full graph per 1 to 10 swap layers, for total of 10x graphs
    f"{idx}_100nodes_1swap_layers.json",
    f"{idx}_100nodes_2swap_layers.json",
    f"{idx}_100nodes_3swap_layers.json",
    f"{idx}_100nodes_4swap_layers.json",
    f"{idx}_100nodes_5swap_layers.json",
    f"{idx}_100nodes_6swap_layers.json",
    f"{idx}_100nodes_7swap_layers.json",
    f"{idx}_100nodes_8swap_layers.json",
    f"{idx}_100nodes_9swap_layers.json",
    f"{idx}_100nodes_10swap_layers.json",
    # 1x 100-node random-regular graph per degree d=3,4,5,6,7,8,9, for a total of 7x graphs
    f"{idx}_100nodes_random3regular.json",
    f"{idx}_100nodes_random4regular.json",
    f"{idx}_100nodes_random5regular.json",
    f"{idx}_100nodes_random6regular.json",
    f"{idx}_100nodes_random7regular.json",
    f"{idx}_100nodes_random8regular.json",
    f"{idx}_100nodes_random9regular.json",
}

chosen_instances: dict[EvaluationType, set[GraphKey] | dict[bool, set[str]]] = {
    "SV": SMALL_INSTANCES,
    "PP": LARGE_INSTANCES,
    "MPS": {False: LARGE_INSTANCES, True: LARGE_INSTANCES},
}
ALL_CHOSEN_INSTANCES = SMALL_INSTANCES.union(LARGE_INSTANCES)

In [3]:
def ignore_these_files(filename: str, results: dict[str, Any]) -> bool:
    """Ignore Parameter Transfer and unoptimised Linear Ramp results for this figure."""

    # If we have Parameter Transfer, ignore.
    if "PT_" in filename:
        return True

    # If we have Linear Ramp without angle optimisation, ignore. This resolves
    # some bond-dimension inconsistencies between _opt and _angle_opt LR
    # results. These are instead extracted from _angle_opt data.
    if (
        "_LR_" in filename
        and "_opt" in filename.lower()
        and not ("angle_opt" in filename.lower() or "angleopt" in filename.lower())
    ):
        # We have LR without angle optimisation
        if "angle" in filename:
            print(f"WARNING: Ignoring {filename!r}")
        return True
    return False


db_filename: str | None
db_filename = None
if db_filename is not None:
    # If we're loading from a saved file.
    db = database.ResultsDatabase(db_filename)
else:
    db = db = database.ResultsDatabase()
    # Add date
    for _folder in tqdm(
        [
            "../../data/training/random_regular",
            "../../data/training/heavy_hex",
            "../../data/training/line_to_full",
            "../../data/training/erdos_renyi",
        ]
    ):
        db.add_data(_folder, ignore_file_function=ignore_these_files)
db = db.filter_by(instance_filter=chosen_instances)

  0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# These are all methods present in the data.
db.print_methods_by_evaluation()

       MPS (Aer)         |      MPS (Quimb)      |          PP          |          SV         
----------------------------------------------------------------------------------------------
FA_MPSAer_no_opt.json    | FA_MPS_no_opt.json    | FA_PP_no_opt.json    | FA_SV_no_opt.json   
FA_MPSAer_opt.json       | FA_MPS_opt.json       | FA_PP_opt.json       | FA_SV_opt.json      
F_MPSAer.json            | F_MPS.json            | F_PP.json            | F_SV.json           
I_MPSAer.json            | I_MPS.json            | I_PP.json            | I_SV.json           
LR_MPSAer_angle_opt.json | LR_MPS_angle_opt.json | LR_PP_angle_opt.json | LR_SV_angle_opt.json
LR_MPSAer_opt.json       | LR_MPS_opt.json       | LR_PP_opt.json       | LR_SV_opt.json      
TQA_MPSAer_no_opt.json   | RTS_MPS.json          | RTS_PP.json          | TQA_SV_no_opt.json  
TQA_MPSAer_opt.json      | TQA_MPS_no_opt.json   | TQA_PP_no_opt.json   | TQA_SV_opt.json     
                         | TQA_MPS_opt.json      |

## Best-Methods Figure


### Get Best-Methods Dataframe


In [5]:
# Get best methods per instance, i.e., instance-depth pairs. As we already
# filtered by the instances, this will ensure we only have the best-method per
# instance in `chosen_instances`.

# We ignore the warnings about missing min-max cut data. We didn't add those
# files to the database as the best-methods figure doesn't use approximation
# ratios AND we can determine the best methods using the energies.
warnings.filterwarnings("ignore", "Missing min-max cut data for instance")

# Create the best-methods dataframe.
df_best = db.only_best_parameters("instance").to_dataframe()
df_best

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,...,metadata,result_index,run_datetime,result_key_index,approximation_ratio,mps_bond_dimension,mps_threshold,pp_max_weight,pp_min_abs_coeff,fa_degree
0,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,10,10.380828,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-01 15:07:45,1.0,NaN,NaN,NaN,NaN,NaN,None
1,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,1,5.156636,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 17:13:56,1.0,NaN,NaN,NaN,NaN,NaN,None
2,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,3,8.330960,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 20:54:55,1.0,NaN,NaN,NaN,NaN,NaN,None
3,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,4,9.059836,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 21:00:11,1.0,NaN,NaN,NaN,NaN,NaN,None
4,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,5,9.528705,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 21:04:04,1.0,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_angle_opt.json,LR_angle_opt.json,7,11.660903,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 10:43:35,1.0,NaN,NaN,NaN,NaN,NaN,None
1476,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_angle_opt.json,LR_angle_opt.json,8,11.838120,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 11:04:57,1.0,NaN,NaN,NaN,NaN,NaN,None
1477,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_angle_opt.json,LR_angle_opt.json,10,12.257593,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 11:57:49,1.0,NaN,NaN,NaN,NaN,NaN,None
1478,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,FA_SV_opt.json,FA_opt.json,4,10.342585,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-02 16:33:30,1.0,NaN,NaN,NaN,NaN,NaN,None


### Plot Figure


In [6]:
# Custom colours taken from Paul Tol's bright palette.
custom_palette = [
    "#4477AA",
    "#EE6677",
    "#228833",
    "#CCBB44",
    "#66CCEE",
    "#AA3377",
    "#000000",
]

In [15]:
# Reduce font size to match the paper.
plt.rcParams["legend.fontsize"] = "small"
plt.rcParams["axes.titlesize"] = "small"
plt.rcParams["axes.labelsize"] = "small"
plt.rcParams["figure.labelsize"] = "small"
plt.rcParams["xtick.labelsize"] = "small"
plt.rcParams["ytick.labelsize"] = "small"

PROPORTION = True
"""If the figure shows the proportion of instances per method (``True``) or the sum (``False``)."""

fig, axes = plt.subplots(1, 4, figsize=(7, 2.5), sharey=True, dpi=300)
handles, labels = None, None
palette = custom_palette

# We only label methods which are best at least once.
training_methods = list(str(x) for x in sorted(df_best["method_label"].unique()))

for ax, (evaluation, with_aer) in zip(
    axes.flatten(), [("SV", False), ("PP", False), ("MPS", False), ("MPS", True)]
):
    # Set titles, x-ticks, and x limits.
    ax.set_title(
        f"{evaluation} {('(Aer)' if with_aer else '(Quimb)') if evaluation == 'MPS' else ''}"
    )
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1, offset=1))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.set_xlim(1.5, 10.5)

    # Get the data for this evaluation method.
    # We use cast() so we don't get type-hint errors.
    sub_data = cast(
        pd.DataFrame,
        df_best[
            (df_best["evaluation"] == evaluation)
            & (df_best["with_aer"] == with_aer)
            & (df_best["depth"] > 1)
        ],
    )

    # Print error message if there are no results.
    if len(sub_data) == 0:
        print(f"No data for evaluation={evaluation} and with_aer={with_aer}")
        continue

    # Plot using Seaborn as matplotlib doesn't seem to support stacked bars.
    _ = sns.histplot(
        data=sub_data,
        x="depth",
        hue="method_label",
        multiple="fill" if PROPORTION else "stack",
        stat="proportion" if PROPORTION else "count",
        ax=ax,
        # Use bins that are offset, so we catch the correct depths.
        bins=np.arange(1.5, 11, 1),
        hue_order=training_methods,
        legend=False,
        palette=palette,
    )

    # Remove xlabel and ylabel as we use supxlabel and supylabel. We do this
    # after sns.histplot as Seaborn also sets the axis labels.
    ax.set_xlabel("")
    ax.set_ylabel("")

# Set figure labels.
fig.supylabel("Proportion" if PROPORTION else "Count")
fig.supxlabel("Depth $p$", y=0.1)

# Create legend.
legend_entries = [
    (
        Patch(facecolor=hue, edgecolor="k"),
        utils.labels.format_method_label_to(label, "latex"),
    )
    for hue, label in zip(
        sns.color_palette(palette) if isinstance(palette, str) else palette,
        training_methods,
    )
]
fig.legend(
    *zip(*legend_entries),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.1, 0, 0),
    ncol=6,
    frameon=False,
)
plt.tight_layout()
plt.savefig(
    f"best_methods{'_counts' if not PROPORTION else ''}.pdf", bbox_inches="tight"
)

/tmp/ipykernel_216708/39700616.py:47: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  _ = sns.histplot(
/tmp/ipykernel_216708/39700616.py:47: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  _ = sns.histplot(
/tmp/ipykernel_216708/39700616.py:47: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  _ = sns.histplot(
/tmp/ipykernel_216708/39700616.py:47: UserWarning: The palette list has more values (7) than needed (6), which may not be intended.
  _ = sns.histplot(


## Get Missing Configurations


In [8]:
# Failed runs are those which did not return a valid results JSON file. We mark
# why in in `failed_runs.json`
failed_runs_filename = Path("failed_runs_best_methods.json")
failed_runs: dict[GraphKey, dict[MethodConfigJSON, dict[Depth, str]]]
if failed_runs_filename.exists():
    failed_runs = db.load_failed_configs_from_json(failed_runs_filename)
else:
    failed_runs = defaultdict(lambda: defaultdict(dict))


In [9]:
# These are the methods we would like to include in our figure. Some are
# 'virtual' _no_opt methods as no method JSON file exists, but we extract the
# data from an intermediate run of an _opt-method run.
target_methods: dict[
    EvaluationType, list[MethodConfigJSON] | dict[bool, list[MethodConfigJSON]]
] = {
    "MPS": {
        True: cast(
            list[MethodConfigJSON],
            [
                "FA_MPSAer_no_opt.json",
                "FA_MPSAer_opt.json",
                "F_MPSAer.json",
                "I_MPSAer.json",
                "LR_MPSAer_opt.json",
                "LR_MPSAer_angle_opt.json",
                "TQA_MPSAer_no_opt.json",
                "TQA_MPSAer_opt.json",
            ],
        ),
        False: cast(
            list[MethodConfigJSON],
            [
                "FA_MPS_no_opt.json",
                "FA_MPS_opt.json",
                "F_MPS.json",
                "I_MPS.json",
                "LR_MPS_opt.json",
                "LR_MPS_angle_opt.json",
                "RTS_MPS.json",
                "TQA_MPS_no_opt.json",
                "TQA_MPS_opt.json",
            ],
        ),
    },
    "PP": cast(
        list[MethodConfigJSON],
        [
            "FA_PP_no_opt.json",
            "FA_PP_opt.json",
            "F_PP.json",
            "I_PP.json",
            "LR_PP_angle_opt.json",
            "LR_PP_opt.json",
            # We don't want Parameter Transfer at all.
            # "PT_PP_AAAM.json",
            "RTS_PP.json",
            "TQA_PP_no_opt.json",
            "TQA_PP_opt.json",
        ],
    ),
    "SV": cast(
        list[MethodConfigJSON],
        [
            "FA_SV_no_opt.json",
            "FA_SV_opt.json",
            "F_SV.json",
            "I_SV.json",
            "LR_SV_angle_opt.json",
            "TQA_SV_no_opt.json",
            "TQA_SV_opt.json",
            "TS_SV.json",
        ],
    ),
}

all_methods: list[MethodConfigJSON] = list(
    set(
        method
        for methods in target_methods.values()
        for method in (
            methods
            if isinstance(methods, list)
            else [m for sublist in methods.values() for m in sublist]
        )
    )
)

In [10]:
# Get missing configs. We make the database's life easier by only storing the
# best-parameters per config, i.e., the best energy per instance-method-depth
# tuples.
missing_configs: dict[
    EvaluationType | tuple[Literal["MPS"], bool],
    dict[MethodConfigJSON, dict[Depth, set[GraphKey]]],
] = db.only_best_parameters("config").get_missing_configs(
    target_methods=target_methods,
    target_instances=chosen_instances,
    target_depths=range(2, 11),
    failed_configs=failed_runs,
    with_derived_configs=False,
)

# Convert the missing configs into a dataframe.
db_missing_configs_data = []
for eval_key, method_dict in missing_configs.items():
    if isinstance(eval_key, tuple):
        evaluation, with_aer = eval_key
    else:
        evaluation = eval_key
        with_aer = None

    for method_config, depth_dict in method_dict.items():
        for depth, instance_set in depth_dict.items():
            for instance_name in instance_set:
                db_missing_configs_data.append(
                    {
                        # Helper fields
                        "evaluation": evaluation,
                        "with_aer": with_aer,
                        "graph_type": utils.instance.graph_type(instance_name),
                        "evaluation_label": utils.labels.trainer_config_to_evaluation_label(
                            method_config
                        ),
                        "method_label": utils.labels.trainer_config_to_method_label(
                            method_config
                        ),
                        # Config
                        "method": method_config,
                        "depth": depth,
                        "instance": instance_name,
                    }
                )

df_missing_configs = pd.DataFrame(db_missing_configs_data)
if not df_missing_configs.empty:
    df_missing_configs = df_missing_configs.set_index(
        ["evaluation", "with_aer", "method", "depth"]
    )
df_missing_configs = df_missing_configs.reset_index().sort_values(
    ["evaluation", "with_aer", "method", "depth", "instance"]
)
# Save to a csv for easier referencing
df_missing_configs.to_csv("missing_best_method_configs.csv")
df_missing_configs

,evaluation,with_aer,method,depth,graph_type,evaluation_label,method_label,instance
0,MPS,True,F_MPSAer.json,10,line_to_full,MPS (Aer),Fourier*,001_100nodes_10swap_layers.json
1,MPS,True,I_MPSAer.json,10,line_to_full,MPS (Aer),Interp.*,001_100nodes_10swap_layers.json


### Missing Configurations Table


In [11]:
# Create multi-index of target evaluation and method labels.
row_index = pd.MultiIndex.from_tuples(
    [
        (
            evaluation_label,
            method_label,
        )
        for evaluation_label, method_label in product(
            ["MPS (Aer)", "MPS (Quimb)", "PP", "SV"],
            sorted(
                set(
                    [
                        utils.labels.trainer_config_to_method_label(m)
                        for m in all_methods
                    ]
                )
            ),
        )
    ],
)

In [12]:
pivot_missing = df_missing_configs.pivot_table(
    values="instance",
    index=["evaluation_label", "method_label"],
    columns=["graph_type", "depth"],
    aggfunc="count",
)
pivot_missing = pivot_missing.reindex(index=row_index)
display("Missing Runs for Best-Methods Figure")
display(
    pivot_missing.style.format(precision=0, na_rep=" ")
    .background_gradient("RdYlGn_r", axis=None)  # pyright: ignore[reportAttributeAccessIssue]
    .highlight_null("transparent")
)

'Missing Runs for Best-Methods Figure'